# Week 7: K-means Clustering Sweep and Elbow Analysis

This notebook evaluates K-means on the autoencoder embeddings (13 dimensions) using a parameter sweep for $k = 2..20$.

It produces:
- a full inertia table
- a full silhouette table
- an elbow curve
- a silhouette curve
- a final recommendation for the best $k$

The goal is to support the Week 7 clustering report with reproducible evidence.

## Input data

Loads the autoencoder embeddings (13 dimensions) generated in the previous step.

In [1]:
from pathlib import Path

import json

import numpy as np
import pandas as pd
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

project_root = Path.cwd()
if not (project_root / 'data').exists():
    project_root = project_root.parent
if not (project_root / 'data').exists():
    project_root = project_root.parent

ARTIFACTS_DIR = project_root / 'artifacts' / 'week07'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

input_path = ARTIFACTS_DIR / 'week07_autoencoder_embeddings_latent_13.parquet'
input_path

PosixPath('/Users/jay17/Documents/Proyects/big-data-tf/artifacts/week07/week07_autoencoder_embeddings_latent_13.parquet')

## Load the autoencoder embeddings

This matrix is the clustering input. We keep `movieId` as an identifier and cluster on the embedding dimensions.

In [2]:
embedding_frame = pl.read_parquet(input_path)
if 'movieId' not in embedding_frame.columns:
    raise ValueError('Expected a movieId column in the autoencoder embeddings file.')

embedding_columns = [column for column in embedding_frame.columns if column != 'movieId']
if len(embedding_columns) < 2:
    raise ValueError('Need at least two embedding dimensions to run clustering.')

X = embedding_frame.select(embedding_columns).to_pandas().astype(float)
movie_ids = embedding_frame.get_column('movieId').to_list()

print(f'Loaded {embedding_frame.height:,} movies with {len(embedding_columns)} autoencoder embedding dimensions from {input_path}')
display(embedding_frame.head(5).to_pandas())

Loaded 62,423 movies with 13 autoencoder embedding dimensions from /Users/jay17/Documents/Proyects/big-data-tf/artifacts/week07/week07_autoencoder_embeddings_latent_13.parquet


,movieId,ae_1,ae_2,ae_3,ae_4,ae_5,ae_6,ae_7,ae_8,ae_9,ae_10,ae_11,ae_12,ae_13
0,1,-4.406366,-1.893490,-4.860766,10.483939,-2.952837,-6.017283,1.625875,5.934033,1.859101,-1.943249,1.329634,-1.781883,1.263428
1,2,-1.525542,2.367595,-1.534571,8.508888,-4.085185,-7.362682,-2.897106,2.801864,2.670520,2.836757,1.098903,-0.138598,-0.802121
2,3,-1.760156,0.054254,-0.914767,0.844683,-1.185236,-5.744865,4.125343,2.357876,2.513872,-3.528446,-1.466366,-3.248106,-2.132906
3,4,-1.205828,4.405512,2.952558,-2.593609,-1.653408,-6.934993,1.503002,-3.399956,1.053031,0.764881,3.978532,-0.204923,-2.822405
4,5,-2.604524,1.899337,-2.099546,3.412642,-3.847823,-2.465474,2.383733,5.084182,3.629400,-0.279647,-0.533422,-1.083439,2.463400


## K-means parameter sweep

We sweep $k$ from 2 to 20 and record inertia and silhouette score for each run.

The full table is the main evidence for deciding the best cluster count.

In [3]:
random_state = 42
k_values = list(range(2, 21))
results = []
labels_by_k = {}

for k in k_values:
    model = KMeans(n_clusters=k, random_state=random_state, n_init=10)
    labels = model.fit_predict(X)
    inertia = float(model.inertia_)
    silhouette = float(silhouette_score(X, labels))
    labels_by_k[k] = labels

    print(f'  k={k}: inertia={inertia:.4f}, silhouette={silhouette:.4f}, cluster_sizes=[{int(pd.Series(labels).value_counts().min())}-{int(pd.Series(labels).value_counts().max())}]')

    results.append({
        'k': k,
        'inertia': inertia,
        'silhouette': silhouette,
        'cluster_size_min': int(pd.Series(labels).value_counts().min()),
        'cluster_size_max': int(pd.Series(labels).value_counts().max()),
    })

metrics_df = pd.DataFrame(results)
metrics_df['inertia_drop'] = metrics_df['inertia'].shift(1) - metrics_df['inertia']
metrics_df['silhouette_delta'] = metrics_df['silhouette'].diff()

metrics_path = ARTIFACTS_DIR / 'week07_kmeans_metrics.csv'
metrics_df.to_csv(metrics_path, index=False)

display(metrics_df)
print(f'Saved metrics to {metrics_path}')

  k=2: inertia=3337464.0009, silhouette=0.3312, cluster_sizes=[5639-56784]
  k=3: inertia=3054836.1077, silhouette=0.1699, cluster_sizes=[5628-37031]
  k=4: inertia=2858886.6901, silhouette=0.1564, cluster_sizes=[5595-29266]
  k=5: inertia=2724017.7249, silhouette=0.1593, cluster_sizes=[5241-26611]
  k=6: inertia=2603019.3812, silhouette=0.1651, cluster_sizes=[3932-25650]
  k=7: inertia=2504220.8215, silhouette=0.1721, cluster_sizes=[2995-25183]
  k=8: inertia=2412340.9399, silhouette=0.1204, cluster_sizes=[2981-14542]
  k=9: inertia=2362981.3756, silhouette=0.1220, cluster_sizes=[1807-13290]
  k=10: inertia=2283938.1389, silhouette=0.1277, cluster_sizes=[2867-13550]
  k=11: inertia=2225336.6694, silhouette=0.1284, cluster_sizes=[1398-12446]
  k=12: inertia=2170169.0283, silhouette=0.1343, cluster_sizes=[1396-11962]
  k=13: inertia=2116419.3341, silhouette=0.1371, cluster_sizes=[1396-12225]
  k=14: inertia=2075630.4633, silhouette=0.1360, cluster_sizes=[1391-10842]
  k=15: inertia=2044

,k,inertia,silhouette,cluster_size_min,cluster_size_max,inertia_drop,silhouette_delta
0,2,3.337464e+06,0.331156,5639,56784,NaN,NaN
1,3,3.054836e+06,0.169941,5628,37031,282627.893220,-0.161216
2,4,2.858887e+06,0.156393,5595,29266,195949.417545,-0.013547
3,5,2.724018e+06,0.159301,5241,26611,134868.965245,0.002907
4,6,2.603019e+06,0.165066,3932,25650,120998.343672,0.005766
5,7,2.504221e+06,0.172062,2995,25183,98798.559693,0.006996
6,8,2.412341e+06,0.120441,2981,14542,91879.881680,-0.051621
7,9,2.362981e+06,0.121988,1807,13290,49359.564235,0.001547
8,10,2.283938e+06,0.127689,2867,13550,79043.236677,0.005701
9,11,2.225337e+06,0.128446,1398,12446,58601.469498,0.000757


Saved metrics to /Users/jay17/Documents/Proyects/big-data-tf/artifacts/week07/week07_kmeans_metrics.csv


## Elbow and silhouette plots

The elbow curve shows inertia across $k$, while the silhouette curve shows separation quality. Both should be read together.

In [4]:
fig_elbow = go.Figure()

fig_elbow.add_trace(
    go.Scatter(
        x=metrics_df['k'],
        y=metrics_df['inertia'],
        mode='lines+markers',
        name='Inertia',
        line=dict(color='#1f77b4', width=3),
    )
)

fig_elbow.update_layout(
    title='Elbow Curve: Inertia vs k',
    xaxis_title='k',
    yaxis_title='Inertia',
    template='plotly_white',
    width=800,
    height=500,
)

elbow_path = ARTIFACTS_DIR / 'week07_kmeans_elbow.html'

fig_elbow.write_html(str(elbow_path))
fig_elbow.write_image(
    str(elbow_path.with_suffix('.png')),
    width=800,
    height=500,
    scale=2,
)

fig_elbow.show()


fig_silhouette = go.Figure()

fig_silhouette.add_trace(
    go.Scatter(
        x=metrics_df['k'],
        y=metrics_df['silhouette'],
        mode='lines+markers',
        name='Silhouette',
        line=dict(color='#d62728', width=3),
    )
)

fig_silhouette.update_layout(
    title='Silhouette Score vs k',
    xaxis_title='k',
    yaxis_title='Silhouette score',
    template='plotly_white',
    width=800,
    height=500,
)

silhouette_path = ARTIFACTS_DIR / 'week07_kmeans_silhouette.html'

fig_silhouette.write_html(str(silhouette_path))
fig_silhouette.write_image(
    str(silhouette_path.with_suffix('.png')),
    width=800,
    height=500,
    scale=2,
)

fig_silhouette.show()

print(
    f"Saved plots to:\n"
    f"{elbow_path}\n"
    f"{silhouette_path}"
)

Saved plots to:
/Users/jay17/Documents/Proyects/big-data-tf/artifacts/week07/week07_kmeans_elbow.html
/Users/jay17/Documents/Proyects/big-data-tf/artifacts/week07/week07_kmeans_silhouette.html


## Decide the best k

Use the table and plots together.

A simple default choice is the k with the highest silhouette score, then check whether that choice also sits near the elbow.

In [5]:
filtered_metrics = metrics_df[metrics_df['k'] != 2]

best_row = filtered_metrics.loc[filtered_metrics['silhouette'].idxmax()]
best_k = int(best_row['k'])
best_k_inertia = float(best_row['inertia'])
best_k_silhouette = float(best_row['silhouette'])

final_model = KMeans(n_clusters=best_k, random_state=random_state, n_init=10)
final_labels = final_model.fit_predict(X)

cluster_sizes = (
    pd.Series(final_labels)
    .value_counts()
    .sort_index()
    .rename_axis('cluster')
    .reset_index(name='count')
)

assignments = pd.DataFrame({
    'movieId': movie_ids,
    'kmeans_cluster': final_labels,
})

assignments_path = ARTIFACTS_DIR / 'week07_kmeans_assignments.csv'
assignments.to_csv(assignments_path, index=False)

print(f'Best k by silhouette (excluding k=2): {best_k}')
print(f'Inertia at best k: {best_k_inertia:.4f}')
print(f'Silhouette at best k: {best_k_silhouette:.4f}')
display(cluster_sizes)
print(f'Saved cluster assignments to {assignments_path}')

Best k by silhouette (excluding k=2): 7
Inertia at best k: 2504220.8215
Silhouette at best k: 0.1721


,cluster,count
0,0,9364
1,1,2995
2,2,5362
3,3,10458
4,4,25183
5,5,5497
6,6,3564


Saved cluster assignments to /Users/jay17/Documents/Proyects/big-data-tf/artifacts/week07/week07_kmeans_assignments.csv


In [6]:
df_clustered = embedding_frame.clone().to_pandas()
df_clustered["cluster"] = final_labels

In [7]:
numeric_cols = [c for c in df_clustered.columns if c != "movieId"]
numeric_summary = df_clustered.groupby("cluster")[numeric_cols].mean().round(3)
display(numeric_summary)

,ae_1,ae_2,ae_3,ae_4,ae_5,ae_6,ae_7,ae_8,ae_9,ae_10,ae_11,ae_12,ae_13,cluster
cluster,,,,,,,,,,,,,,
0,0.418,0.050,2.128,-0.143,-0.295,-2.873,-0.589,1.282,0.889,-1.327,-1.515,-0.464,-2.264,0.0
1,-1.686,1.364,0.999,-2.371,4.943,-1.176,-1.886,0.530,-2.233,0.396,-3.005,1.397,-3.824,1.0
2,-1.583,-1.504,4.561,0.718,0.788,0.123,0.175,-2.776,-1.711,0.695,-3.984,0.535,-1.713,2.0
3,0.107,0.802,2.862,1.620,1.584,0.240,-3.040,-0.347,-1.381,0.736,-2.059,-1.554,-0.739,3.0
4,1.569,-0.547,1.507,0.106,0.721,0.214,-0.622,-0.037,-1.692,0.465,-0.837,1.350,-2.212,4.0
5,-3.309,1.218,-3.393,2.614,4.764,3.631,-1.106,4.778,-1.844,1.216,-5.773,3.974,-3.225,5.0
6,0.539,-2.312,1.692,4.611,2.053,0.164,-1.927,0.335,-1.846,-3.087,-0.095,1.127,-4.161,6.0


# Cluster Interpretation Analysis

This section maps the K-means cluster assignments back to the original movie metadata to produce interpretable summaries (sizes, numeric feature statistics, genre proportions, distinctive genres, representative movies) and a short semantic label for each cluster.

Key: We analyze the **top-20 one-hot encoded genres** and **raw rating/count data** (already normalized in preprocessing).

In [12]:
# Cluster interpretation using the same one-hot matrix used by the autoencoder notebook
from pathlib import Path
import json as _json
import numpy as np
import pandas as pd
import polars as pl

_here = Path.cwd()
project_root = next(
    (p for p in [_here, _here.parent, _here.parent.parent] if (p / 'artifacts').exists()),
    _here
)

# 1) Load Week 5 one-hot feature matrix (same source used by autoencoder training)
feature_candidates = [
    project_root / 'artifacts' / 'week05' / 'week05_pca_feature_matrix.parquet',
    Path('/artifacts/week05/week05_pca_feature_matrix.parquet'),
]
feature_path = next((p for p in feature_candidates if p.exists()), None)
if feature_path is None:
    raise FileNotFoundError('Could not find week05_pca_feature_matrix.parquet in artifacts/week05 or /content')

feature_frame = pl.read_parquet(feature_path).to_pandas()
print(f'Loaded feature frame from {feature_path} with {feature_frame.shape[0]:,} rows')
print(f'Columns: {list(feature_frame.columns)[:20]}...')

# 2) Denormalize rating statistics from Week 5 features
# Week 5 used log1p for count and z-score for mean rating (mean~3.5, std~1.0)
rating_count = np.expm1(feature_frame['rating_count_log']) if 'rating_count_log' in feature_frame.columns else feature_frame.get('rating_count', np.nan)
rating_mean = (feature_frame['avg_rating_z'] * 1.0 + 3.5) if 'avg_rating_z' in feature_frame.columns else feature_frame.get('rating_mean', np.nan)
rating_std = feature_frame['rating_std'] if 'rating_std' in feature_frame.columns else np.nan

processed_stats = pd.DataFrame({
    'movieId': feature_frame['movieId'],
    'rating_mean': rating_mean,
    'rating_count': rating_count,
    'rating_std': rating_std,
})
print(f'Denormalized rating stats from Week 5: {processed_stats.shape[0]:,} rows')

# 3) Load cluster assignments
assign_path = project_root / 'artifacts' / 'week07' / 'week07_kmeans_assignments.csv'
if not assign_path.exists():
    raise FileNotFoundError(f'Cluster assignments not found: {assign_path}')

clustered = pd.read_csv(assign_path)
if 'movieId' not in clustered.columns:
    clustered = clustered.rename(columns={clustered.columns[0]: 'movieId'})
if 'kmeans_cluster' in clustered.columns:
    clustered = clustered.rename(columns={'kmeans_cluster': 'cluster'})
print(f'Loaded cluster assignments for {clustered.shape[0]:,} movies')

# 4) Merge stats + clusters + one-hot genre/tag columns
merged = processed_stats.merge(clustered[['movieId', 'cluster']], on='movieId', how='inner')
prefixed_cols = [c for c in feature_frame.columns if str(c).startswith('genre_') or str(c).startswith('tag_')]
if prefixed_cols:
    merged = merged.merge(feature_frame[['movieId'] + prefixed_cols], on='movieId', how='left')
print(f'Merged data: {merged.shape[0]:,} rows with denormalized stats + genres + tags + clusters')

# 5) Cluster sizes
cluster_counts = merged['cluster'].value_counts().sort_index().rename_axis('cluster').reset_index(name='count')
cluster_counts['percentage'] = (cluster_counts['count'] / cluster_counts['count'].sum() * 100).round(2)
cluster_counts_path = project_root / 'artifacts' / 'week07' / 'week07_cluster_sizes.csv'
cluster_counts.to_csv(cluster_counts_path, index=False)
print('Cluster sizes saved to', cluster_counts_path)

# 6) Numeric summary
numeric_cols = ['rating_mean', 'rating_count', 'rating_std']
numeric_summary = merged.groupby('cluster')[numeric_cols].agg(['mean', 'median', 'std', 'min', 'max']).round(3)
numeric_summary.columns = ['_'.join(col).strip() for col in numeric_summary.columns.values]
numeric_summary_path = project_root / 'artifacts' / 'week07' / 'week07_cluster_numeric_summary.csv'
numeric_summary.reset_index().to_csv(numeric_summary_path, index=False)
print('Numeric summary saved to', numeric_summary_path)

global_means = merged[numeric_cols].mean()
cluster_means = merged.groupby('cluster')[numeric_cols].mean()
diff_from_global = (cluster_means - global_means).round(3)
diff_path = project_root / 'artifacts' / 'week07' / 'week07_cluster_numeric_diff_from_global.csv'
diff_from_global.reset_index().to_csv(diff_path, index=False)
print('Numeric differences from global mean saved to', diff_path)

def _onehot_indicator_columns(df, prefix):
    cols = []
    for c in df.columns:
        if not str(c).startswith(prefix):
            continue
        s = df[c]
        if not pd.api.types.is_numeric_dtype(s):
            continue
        non_null = s.dropna()
        if non_null.empty:
            continue
        # Keep only one-hot/proportion-like columns and exclude logs/counts/spans.
        if ((non_null >= 0) & (non_null <= 1)).all():
            cols.append(c)
    return cols

# 7) Genre/tag proportions from existing one-hot matrix (no parsing)
genre_cols = _onehot_indicator_columns(merged, 'genre_')
tag_cols = _onehot_indicator_columns(merged, 'tag_')
print(f'Found {len(genre_cols)} one-hot/proportion genre columns and {len(tag_cols)} one-hot/proportion tag columns')

if not genre_cols:
    print('Warning: no one-hot genre columns found in week05_pca_feature_matrix.parquet')
if not tag_cols:
    print('Warning: no one-hot tag columns found in week05_pca_feature_matrix.parquet')

if genre_cols:
    genre_prop = merged.groupby('cluster')[genre_cols].mean().round(3)
    genre_prop_path = project_root / 'artifacts' / 'week07' / 'week07_cluster_genre_proportions.csv'
    genre_prop.reset_index().to_csv(genre_prop_path, index=False)
    print('Genre proportions saved to', genre_prop_path)
    top5_genres = {}
    for cl in genre_prop.index:
        top = genre_prop.loc[cl].sort_values(ascending=False).head(5)
        top5_genres[int(cl)] = list(zip(top.index.tolist(), top.values.round(3).tolist()))
else:
    genre_prop = pd.DataFrame()
    top5_genres = {}

if tag_cols:
    tag_prop = merged.groupby('cluster')[tag_cols].mean().round(3)
    tag_prop_path = project_root / 'artifacts' / 'week07' / 'week07_cluster_tag_proportions.csv'
    tag_prop.reset_index().to_csv(tag_prop_path, index=False)
    print('Tag proportions saved to', tag_prop_path)
    top5_tags = {}
    for cl in tag_prop.index:
        top = tag_prop.loc[cl].sort_values(ascending=False).head(5)
        top5_tags[int(cl)] = list(zip(top.index.tolist(), top.values.round(3).tolist()))
else:
    tag_prop = pd.DataFrame()
    top5_tags = {}

# 8) Distinctiveness (cluster proportion minus global proportion)
if genre_cols:
    global_genre = merged[genre_cols].mean()
    distinctive_genre = (genre_prop - global_genre).round(3)
    distinctive_genre_path = project_root / 'artifacts' / 'week07' / 'week07_cluster_genre_distinctiveness.csv'
    distinctive_genre.reset_index().to_csv(distinctive_genre_path, index=False)
    top5_distinctive_genre = {}
    for cl in distinctive_genre.index:
        top = distinctive_genre.loc[cl].sort_values(ascending=False).head(5)
        top5_distinctive_genre[int(cl)] = list(zip(top.index.tolist(), top.values.round(3).tolist()))
else:
    top5_distinctive_genre = {}

if tag_cols:
    global_tag = merged[tag_cols].mean()
    distinctive_tag = (tag_prop - global_tag).round(3)
    distinctive_tag_path = project_root / 'artifacts' / 'week07' / 'week07_cluster_tag_distinctiveness.csv'
    distinctive_tag.reset_index().to_csv(distinctive_tag_path, index=False)
    top5_distinctive_tag = {}
    for cl in distinctive_tag.index:
        top = distinctive_tag.loc[cl].sort_values(ascending=False).head(5)
        top5_distinctive_tag[int(cl)] = list(zip(top.index.tolist(), top.values.round(3).tolist()))
else:
    top5_distinctive_tag = {}

# 9) Representative movies
representative = {}
for cl in sorted(merged['cluster'].unique()):
    sub = merged[merged['cluster'] == cl].copy()
    rand_sample = sub.sample(n=min(10, len(sub)), random_state=42)
    rand_titles = [f"Movie {mid}" for mid in rand_sample['movieId'].tolist()]

    rating_mean_centroid = sub['rating_mean'].mean()
    dists = np.abs(sub['rating_mean'] - rating_mean_centroid)
    idxs = dists.nsmallest(min(10, len(dists)), keep='all').index
    closest = sub.loc[idxs]
    closest_titles = [
        f"Movie {mid} (rating: {rating:.2f})"
        for mid, rating in zip(closest['movieId'].tolist(), closest['rating_mean'].tolist())
    ]

    representative[int(cl)] = {
        'random': rand_titles,
        'closest_by_rating': closest_titles,
    }

# 10) Heuristic semantic labels
cluster_labels = {}
for cl in sorted(merged['cluster'].unique()):
    label_parts = []

    if cl in top5_distinctive_genre and top5_distinctive_genre[cl]:
        label_parts.extend([g.replace('genre_', '').replace('_', ' ').title() for g, _ in top5_distinctive_genre[cl][:2]])
    if cl in top5_distinctive_tag and top5_distinctive_tag[cl]:
        label_parts.extend([t.replace('tag_', '').replace('_', ' ').title() for t, _ in top5_distinctive_tag[cl][:2]])

    cluster_rating = merged.loc[merged['cluster'] == cl, 'rating_mean'].mean()
    global_rating = merged['rating_mean'].mean()
    if cluster_rating > global_rating + 0.1:
        label_parts.append('High-Rated')
    elif cluster_rating < global_rating - 0.1:
        label_parts.append('Lower-Rated')

    label = ' / '.join(label_parts) if label_parts else f'Cluster {cl}'
    description = (
        f"Cluster {cl}: genres={[g for g, _ in top5_distinctive_genre.get(cl, [])[:2]]}; "
        f"tags={[t for t, _ in top5_distinctive_tag.get(cl, [])[:2]]}. "
        f"Mean rating={cluster_rating:.2f}, "
        f"count={merged.loc[merged['cluster'] == cl, 'rating_count'].mean():.0f}."
    )
    cluster_labels[int(cl)] = {'label': label, 'description': description}

# 11) Write markdown report
report_lines = []
report_lines.append('# Cluster Interpretation Report')
report_lines.append('')
report_lines.append('## Data Source Note')
report_lines.append('Rating statistics and one-hot features loaded from Week 5 PCA feature matrix (`week05_pca_feature_matrix.parquet`).')
report_lines.append('')
report_lines.append('## Cluster Overview')
report_lines.append(cluster_counts.to_markdown(index=False))
report_lines.append('')
report_lines.append('## Rating Statistics (denormalized) - Differences from Global Mean')
if not diff_from_global.empty:
    report_lines.append(diff_from_global.to_markdown())
report_lines.append('')
report_lines.append('## Top 5 Genres per Cluster')
for cl, top in top5_genres.items():
    report_lines.append(f'### Cluster {cl} top genres')
    for g, p in top:
        report_lines.append(f'- {g}: {p:.3f}')
    report_lines.append('')

report_lines.append('## Top 5 Tags per Cluster')
for cl, top in top5_tags.items():
    report_lines.append(f'### Cluster {cl} top tags')
    for t, p in top:
        report_lines.append(f'- {t}: {p:.3f}')
    report_lines.append('')

report_lines.append('## Top 5 Distinctive Genres per Cluster')
for cl, top in top5_distinctive_genre.items():
    report_lines.append(f'### Cluster {cl} distinctive genres')
    for g, d in top:
        report_lines.append(f'- {g}: {d:.3f}')
    report_lines.append('')

report_lines.append('## Top 5 Distinctive Tags per Cluster')
for cl, top in top5_distinctive_tag.items():
    report_lines.append(f'### Cluster {cl} distinctive tags')
    for t, d in top:
        report_lines.append(f'- {t}: {d:.3f}')
    report_lines.append('')

report_lines.append('## Representative Movies (random & by rating)')
for cl, vals in representative.items():
    report_lines.append(f'### Cluster {cl} sample movies (random)')
    for t in vals['random']:
        report_lines.append(f'- {t}')
    report_lines.append('')
    report_lines.append(f'### Cluster {cl} sample movies (closest to mean rating)')
    for t in vals['closest_by_rating']:
        report_lines.append(f'- {t}')
    report_lines.append('')

report_lines.append('## Final Cluster Labels (heuristic)')
report_lines.append('| cluster | label | description |')
report_lines.append('|---:|---|---|')
for cl, info in cluster_labels.items():
    report_lines.append(f"| {cl} | {info['label']} | {info['description']} |")

report_path = project_root / 'artifacts' / 'week07' / 'week07_cluster_interpretation.md'
report_path.write_text('\n'.join(report_lines))
print('Wrote cluster interpretation report to', report_path)

# 12) Save representative samples as JSON
with open(project_root / 'artifacts' / 'week07' / 'week07_cluster_representative.json', 'w') as fh:
    _json.dump(representative, fh, indent=2)
print('Wrote representative samples to artifacts/week07/week07_cluster_representative.json')

Loaded feature frame from /Users/jay17/Documents/Proyects/big-data-tf/artifacts/week05/week05_pca_feature_matrix.parquet with 62,423 rows
Columns: ['movieId', 'title', 'genres_list', 'tag_tokens', 'rating_std', 'release_year_z', 'avg_rating_z', 'rating_count_log', 'tag_event_count_log', 'unique_tag_count_log', 'rating_span_seconds_z', 'tag_span_seconds_z', 'genre_count_log', 'genre_drama', 'genre_comedy', 'genre_thriller', 'genre_romance', 'genre_action', 'genre_horror', 'genre_documentary']...
Denormalized rating stats from Week 5: 62,423 rows
Loaded cluster assignments for 62,423 movies
Merged data: 62,423 rows with denormalized stats + genres + tags + clusters
Cluster sizes saved to /Users/jay17/Documents/Proyects/big-data-tf/artifacts/week07/week07_cluster_sizes.csv
Numeric summary saved to /Users/jay17/Documents/Proyects/big-data-tf/artifacts/week07/week07_cluster_numeric_summary.csv
Numeric differences from global mean saved to /Users/jay17/Documents/Proyects/big-data-tf/artifact

In [13]:
# Final check: display Week 7 interpretation artifacts in-notebook
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

project_root = Path.cwd()
project_root = Path("/")
artifacts_dir = project_root / 'artifacts' / 'week07'

files_to_show = {
    'Cluster sizes': artifacts_dir / 'week07_cluster_sizes.csv',
    'Numeric summary': artifacts_dir / 'week07_cluster_numeric_summary.csv',
    'Diff from global': artifacts_dir / 'week07_cluster_numeric_diff_from_global.csv',
    'Genre proportions': artifacts_dir / 'week07_cluster_genre_proportions.csv',
    'Tag proportions': artifacts_dir / 'week07_cluster_tag_proportions.csv',
    'Genre distinctiveness': artifacts_dir / 'week07_cluster_genre_distinctiveness.csv',
    'Tag distinctiveness': artifacts_dir / 'week07_cluster_tag_distinctiveness.csv',
}

for title, path in files_to_show.items():
    print(f'\n=== {title} ===')
    if path.exists():
        df = pd.read_csv(path)
        print(f'{path.name}: {df.shape[0]:,} rows x {df.shape[1]:,} cols')
        display(df.head(10))
    else:
        print(f'Missing: {path}')

report_path = artifacts_dir / 'week07_cluster_interpretation.md'
if report_path.exists():
    print(f'\n=== Interpretation report preview ===')
    report_text = report_path.read_text()
    display(Markdown('\n'.join(report_text.splitlines()[:120])))
else:
    print(f'Missing: {report_path}')


=== Cluster sizes ===
Missing: /artifacts/week07/week07_cluster_sizes.csv

=== Numeric summary ===
Missing: /artifacts/week07/week07_cluster_numeric_summary.csv

=== Diff from global ===
Missing: /artifacts/week07/week07_cluster_numeric_diff_from_global.csv

=== Genre proportions ===
Missing: /artifacts/week07/week07_cluster_genre_proportions.csv

=== Tag proportions ===
Missing: /artifacts/week07/week07_cluster_tag_proportions.csv

=== Genre distinctiveness ===
Missing: /artifacts/week07/week07_cluster_genre_distinctiveness.csv

=== Tag distinctiveness ===
Missing: /artifacts/week07/week07_cluster_tag_distinctiveness.csv
Missing: /artifacts/week07/week07_cluster_interpretation.md


## Week 7 summary

The tables above provide the full parameter sweep evidence for the report.

Use the best silhouette score as the default candidate, then justify the final choice by checking the elbow curve and the cluster-size stability.

In [14]:
!zip -r /artifacts/week07.zip /artifacts/week07

	zip warning: name not matched: /artifacts/week07

zip error: Nothing to do! (try: zip -r /artifacts/week07.zip . -i /artifacts/week07)
